# 🛡️ Hate Speech Detection — CNN-BiLSTM (Indonesian Twitter)

> **Kelompok 12 — PBA 2026** | Dataset: `re_dataset.csv` (~13.169 tweet)

## Mengapa CNN-BiLSTM?

| Kriteria Data | Alasan CNN-BiLSTM Cocok |
|---|---|
| Teks pendek Twitter (~17 kata) | BiLSTM efisien pada sekuens pendek, tidak butuh attention panjang |
| Frasa khas hate speech (n-gram lokal) | Conv1D menangkap trigram/bigram diskriminatif |
| Konteks antar kata penting | Bidirectional LSTM membaca kiri→kanan & kanan→kiri |
| Dataset ~13k (tidak terlalu besar) | ~2.12 juta parameter — cukup kapasitas, tidak overfit |
| Batas < 10 juta parameter | ~2.12 juta parameter ✅ |

```
Embedding → SpatialDropout → Conv1D(128,k=3) → MaxPool
         → BiLSTM(64) → BiLSTM(32) → Dense(128) → Sigmoid
```

## 🗂️ Label (Binary Relevance — 1 model per label)
- **HS** — Hate Speech
- **Abusive** — Bahasa Abusif

## 📁 Struktur Dataset di Kaggle
```
/kaggle/input/<nama-dataset>/
    re_dataset.csv
    new_kamusalay.csv
```

---
## ⚙️ 1. Setup & Konfigurasi

In [ ]:
import os, re, pickle, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# ─── Deteksi environment ─────────────────────────────────────
ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    DATASET_NAME = "hate-speech-id"   # ← sesuaikan dengan nama dataset Anda
    INPUT_DIR    = f"/kaggle/input/{DATASET_NAME}"
    OUTPUT_DIR   = "/kaggle/working"
else:
    INPUT_DIR  = "D:/NLP/data/raw"
    OUTPUT_DIR = "D:/NLP/data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

DATASET_PATH = os.path.join(INPUT_DIR, "re_dataset.csv")
SLANG_PATH   = os.path.join(INPUT_DIR, "new_kamusalay.csv")
TARGET_LABELS = ["HS", "Abusive"]

# ─── Hyperparameter CNN-BiLSTM ───────────────────────────────
MAX_VOCAB        = 15000
MAX_SEQ_LEN      = 50
EMBEDDING_DIM    = 128
CNN_FILTERS      = 128
CNN_KERNEL       = 3
LSTM_UNITS       = 64
DENSE_UNITS      = 128
DROPOUT_RATE     = 0.3
LEARNING_RATE    = 1e-3
EPOCHS           = 20
BATCH_SIZE       = 64
VALIDATION_SPLIT = 0.15
RANDOM_SEED      = 42
MAX_PARAMS       = 10_000_000

np.random.seed(RANDOM_SEED)

print(f"Environment : {'Kaggle ☁️' if ON_KAGGLE else 'Lokal 💻'}")
print(f"Input       : {INPUT_DIR}")
print(f"Output      : {OUTPUT_DIR}")

In [ ]:
import tensorflow as tf

tf.random.set_seed(RANDOM_SEED)
print(f"TensorFlow : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU      : {[g.name for g in gpus]}")
else:
    print("⚠️  GPU tidak terdeteksi — menggunakan CPU")

---
## 📦 2. Load & Preprocessing Data

In [ ]:
STOPWORDS_ID = {
    "yang", "dan", "di", "ke", "dari", "itu", "ini", "dengan", "adalah",
    "ada", "tidak", "juga", "untuk", "pada", "dalam", "sudah", "atau",
    "saya", "aku", "kamu", "dia", "mereka", "kita", "kami", "akan",
    "bisa", "telah", "bahwa", "karena", "oleh", "jadi", "lagi", "ya",
    "jangan", "tapi", "kalau", "mau", "aja", "deh", "sih", "lah",
    "dong", "nih", "kan", "nya", "yg", "dgn", "utk", "rt", "user", "url"
}

def load_slang(path):
    d = pd.read_csv(path, header=None, names=["slang","formal"], encoding="latin-1")
    return dict(zip(d["slang"].str.lower(), d["formal"].str.lower()))

def clean_text(text, slang_dict):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r"\buser\b|\burl\b|\brt\b", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [slang_dict.get(t, t) for t in text.split()]
    tokens = [t for t in tokens if t not in STOPWORDS_ID]
    return " ".join(tokens)

print("Fungsi preprocessing siap ✅")

In [ ]:
print("[+] Load dataset...")
df = pd.read_csv(DATASET_PATH, encoding="latin-1")
print(f"    Shape  : {df.shape}")
print(f"    Kolom  : {list(df.columns)}")

slang_dict = load_slang(SLANG_PATH)
print(f"    Slang  : {len(slang_dict):,} entri")

tweet_col = df.columns[0]
print("[+] Cleaning teks...")
df["clean_text"] = df[tweet_col].apply(lambda t: clean_text(t, slang_dict))

before = len(df)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)
print(f"    Data bersih: {len(df):,} baris (hapus {before-len(df)} kosong)")

df[[tweet_col, "clean_text"] + TARGET_LABELS].head(3)

---
## 📊 3. Eksplorasi Data (EDA)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle("Eksplorasi Dataset Hate Speech Indonesia", fontsize=14, fontweight="bold")
COLORS = ["#4C72B0", "#DD8452"]

# ── Distribusi HS
vc = df["HS"].value_counts().reindex([0,1])
axes[0].bar(["Non-HS","Hate Speech"], vc.values, color=COLORS, edgecolor="white", lw=1.5)
for i,v in enumerate(vc.values):
    axes[0].text(i, v+50, f"{v:,}", ha="center", fontweight="bold")
axes[0].set_title("Label: HS", fontweight="bold")
axes[0].set_ylabel("Jumlah Tweet")
axes[0].set_ylim(0, max(vc.values)*1.15)

# ── Distribusi Abusive
vc2 = df["Abusive"].value_counts().reindex([0,1])
axes[1].bar(["Non-Abusive","Abusive"], vc2.values, color=COLORS, edgecolor="white", lw=1.5)
for i,v in enumerate(vc2.values):
    axes[1].text(i, v+50, f"{v:,}", ha="center", fontweight="bold")
axes[1].set_title("Label: Abusive", fontweight="bold")
axes[1].set_ylim(0, max(vc2.values)*1.15)

# ── Panjang tweet
tweet_len = df["clean_text"].str.split().str.len()
axes[2].hist(tweet_len, bins=30, color="#55A868", edgecolor="white", lw=0.8)
axes[2].axvline(tweet_len.mean(), color="red", ls="--", label=f"Mean: {tweet_len.mean():.1f}")
axes[2].axvline(MAX_SEQ_LEN, color="orange", ls="--", label=f"MAX_SEQ_LEN: {MAX_SEQ_LEN}")
axes[2].set_title("Panjang Tweet (kata)", fontweight="bold")
axes[2].set_xlabel("Jumlah Kata")
axes[2].set_ylabel("Frekuensi")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"eda_overview.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"HS positif   : {df['HS'].mean()*100:.1f}%")
print(f"Abusive pos. : {df['Abusive'].mean()*100:.1f}%")

---
## 🔤 4. Tokenisasi & Padding

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("[+] Fit tokenizer...")
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_text"].tolist())

vocab_size = min(len(tokenizer.word_index) + 1, MAX_VOCAB + 1)
print(f"    Vocabulary size : {vocab_size:,}")

seqs = tokenizer.texts_to_sequences(df["clean_text"].tolist())
X    = pad_sequences(seqs, maxlen=MAX_SEQ_LEN, padding="post", truncating="post")
print(f"    Sequence shape  : {X.shape}")

tok_path = os.path.join(OUTPUT_DIR, "cnn_bilstm_tokenizer.pkl")
with open(tok_path, "wb") as f:
    pickle.dump(tokenizer, f)
print(f"💾 Tokenizer disimpan: {tok_path}")

---
## 🧠 5. Arsitektur Model: CNN-BiLSTM

```
Input(seq_len=50)
  └─ Embedding(vocab, 128)
       └─ SpatialDropout1D(0.3)
            └─ Conv1D(128 filters, kernel=3, relu, same padding)
                 └─ MaxPooling1D(pool=2)
                      └─ BiLSTM(64 units, return_sequences=True)
                           └─ BiLSTM(32 units)
                                └─ Dense(128, relu)
                                     └─ Dropout(0.3)
                                          └─ Dense(1, sigmoid)
```

**Total parameter: ~2.12 juta** ✅ jauh di bawah batas 10 juta.

In [ ]:
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam

# ─── PENTING: buat objek metrik BARU di setiap compile() ─────
# Jika list yang sama dipakai ulang, Keras menambah suffix _1, _2
# → KeyError saat mengakses metrics dict setelah evaluate().
def make_metrics():
    return [
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ]

def get_metric(d, name, default=0.0):
    """Cari metrik; prefix-matching agar tahan suffix _1, _2, …"""
    if name in d: return d[name]
    for k in d:
        if k.startswith(name): return d[k]
    return default

def find_hist_key(hist, base):
    """Cari kunci history; tahan suffix _1, _2, …"""
    if base in hist: return base
    for k in hist:
        if k.startswith(base): return k
    return base


def build_cnn_bilstm(vocab_size, name="CNN_BiLSTM"):
    inp = Input(shape=(MAX_SEQ_LEN,), name="input")

    x = layers.Embedding(vocab_size, EMBEDDING_DIM,
                         mask_zero=False, name="embedding")(inp)
    x = layers.SpatialDropout1D(DROPOUT_RATE)(x)

    # CNN — ekstraksi n-gram / frasa
    x = layers.Conv1D(CNN_FILTERS, CNN_KERNEL,
                      activation="relu", padding="same", name="conv1d")(x)
    x = layers.MaxPooling1D(pool_size=2, name="maxpool")(x)

    # BiLSTM — konteks sekuensial dua arah
    x = layers.Bidirectional(
        layers.LSTM(LSTM_UNITS, return_sequences=True, name="lstm1"),
        name="bilstm1"
    )(x)
    x = layers.Bidirectional(
        layers.LSTM(LSTM_UNITS // 2, name="lstm2"),
        name="bilstm2"
    )(x)

    # Klasifikasi
    x = layers.Dense(DENSE_UNITS, activation="relu", name="dense")(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = Model(inp, out, name=name)
    model.compile(
        optimizer=Adam(LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=make_metrics()   # objek baru
    )
    return model


# Preview arsitektur & jumlah parameter
demo = build_cnn_bilstm(vocab_size, name="cnn_bilstm_demo")
n_params = demo.count_params()
print(f"Total parameter : {n_params:,}")
print(f"Batas maksimum  : {MAX_PARAMS:,}")
print(f"Status          : {'✅ OK' if n_params <= MAX_PARAMS else '❌ MELEBIHI BATAS!'}")
print()
demo.summary()

---
## 🏋️ 6. Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix


def train_label(X, y, label):
    """Train CNN-BiLSTM untuk satu label. Return (model, history, metrics, y_test, y_pred)."""
    tf.random.set_seed(RANDOM_SEED)

    print(f"\n{'='*58}")
    print(f"  TRAINING CNN-BiLSTM — Label: {label}")
    print(f"{'='*58}")

    pos = int(y.sum()); neg = len(y) - pos
    print(f"  Distribusi  : 0={neg:,}, 1={pos:,} ({pos/len(y)*100:.1f}% positif)")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
    )
    print(f"  Train/Test  : {len(X_train):,} / {len(X_test):,}")

    cw  = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weight = {i: float(w) for i,w in enumerate(cw)}
    print(f"  Class weight: {class_weight}")

    model = build_cnn_bilstm(vocab_size, name=f"cnn_bilstm_{label}")

    # Validasi parameter
    n_p = model.count_params()
    print(f"\n  📐 Parameter  : {n_p:,} / {MAX_PARAMS:,}  "
          f"{'✅ OK' if n_p<=MAX_PARAMS else '❌ MELEBIHI!'}")
    if n_p > MAX_PARAMS:
        raise RuntimeError(f"Model melebihi batas {MAX_PARAMS:,} parameter!")

    ckpt = os.path.join(OUTPUT_DIR, f"cnn_bilstm_{label}_best.h5")
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc", patience=5, mode="max",
            restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            ckpt, monitor="val_auc", save_best_only=True, mode="max", verbose=0
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
        ),
    ]

    print(f"\n[+] Training (max_epochs={EPOCHS}, batch={BATCH_SIZE})...")
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=1,
    )

    # Evaluasi
    raw = dict(zip(model.metrics_names, model.evaluate(X_test, y_test, verbose=0)))
    metrics = {
        "accuracy" : get_metric(raw, "accuracy"),
        "auc"      : get_metric(raw, "auc"),
        "precision": get_metric(raw, "precision"),
        "recall"   : get_metric(raw, "recall"),
    }
    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    metrics["f1"] = f1_score(y_test, y_pred)

    print(f"\n  📊 Hasil — {label}")
    print(f"     Accuracy  : {metrics['accuracy']:.4f}")
    print(f"     AUC       : {metrics['auc']:.4f}")
    print(f"     F1-Score  : {metrics['f1']:.4f}")
    print(f"     Precision : {metrics['precision']:.4f}")
    print(f"     Recall    : {metrics['recall']:.4f}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['Non', label])}")
    print(f"  💾 Best model: {ckpt}")

    return model, history, metrics, y_test, y_pred


print("Fungsi training siap ✅")

In [ ]:
available_labels = [l for l in TARGET_LABELS if l in df.columns]

all_histories = {}
all_metrics   = {}
all_preds     = {}

for label in available_labels:
    y = df[label].astype(int).values
    model, history, metrics, y_test, y_pred = train_label(X, y, label)
    all_histories[label] = history.history
    all_metrics[label]   = metrics
    all_preds[label]     = (y_test, y_pred)

print("\n✅ Semua label selesai ditraining!")

---
## 📈 7. Visualisasi Training

In [ ]:
fig, axes = plt.subplots(1, len(available_labels), figsize=(8*len(available_labels), 5))
if len(available_labels) == 1: axes = [axes]
fig.suptitle("CNN-BiLSTM — Training Curves", fontsize=14, fontweight="bold")

for ax, label in zip(axes, available_labels):
    hist  = all_histories[label]
    ep    = range(1, len(hist["loss"]) + 1)
    ax2   = ax.twinx()
    k_auc     = find_hist_key(hist, "auc")
    k_val_auc = find_hist_key(hist, "val_auc")

    l1, = ax.plot(ep,  hist["loss"],      "#4C72B0", lw=2, label="Train Loss")
    l2, = ax.plot(ep,  hist["val_loss"],  "#4C72B0", lw=2, ls="--", label="Val Loss")
    l3, = ax2.plot(ep, hist[k_auc],       "#DD8452", lw=2, label="Train AUC")
    l4, = ax2.plot(ep, hist[k_val_auc],   "#DD8452", lw=2, ls="--", label="Val AUC")

    ax.set_title(f"Label: {label}", fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss", color="#4C72B0")
    ax2.set_ylabel("AUC", color="#DD8452")
    ax.legend(handles=[l1,l2,l3,l4], fontsize=9, loc="center right")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("💾 Training curves disimpan")

In [ ]:
import itertools

fig, axes = plt.subplots(1, len(available_labels), figsize=(6*len(available_labels), 5))
if len(available_labels) == 1: axes = [axes]
fig.suptitle("CNN-BiLSTM — Confusion Matrix", fontsize=14, fontweight="bold")

for ax, label in zip(axes, available_labels):
    y_test, y_pred = all_preds[label]
    cm = confusion_matrix(y_test, y_pred)
    im = ax.imshow(cm, cmap=plt.cm.Blues)
    ax.set_title(f"Label: {label}", fontweight="bold")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Non",label]); ax.set_yticklabels(["Non",label])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    thresh = cm.max() / 2.0
    for i,j in itertools.product(range(2), range(2)):
        ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                color="white" if cm[i,j]>thresh else "black",
                fontsize=14, fontweight="bold")
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 📋 8. Ringkasan Hasil

In [ ]:
rows = []
for label, m in all_metrics.items():
    rows.append({
        "Label"    : label,
        "Model"    : "CNN-BiLSTM",
        "Accuracy" : round(m["accuracy"],  4),
        "AUC"      : round(m["auc"],       4),
        "F1-Score" : round(m["f1"],        4),
        "Precision": round(m["precision"], 4),
        "Recall"   : round(m["recall"],    4),
    })

summary_df = pd.DataFrame(rows)

csv_path = os.path.join(OUTPUT_DIR, "cnn_bilstm_results.csv")
summary_df.to_csv(csv_path, index=False)
print(f"💾 Hasil disimpan: {csv_path}\n")

(
    summary_df.style
    .background_gradient(cmap="YlGn", subset=["Accuracy","AUC","F1-Score"])
    .format(precision=4)
    .set_caption("📊 CNN-BiLSTM — Hasil Evaluasi Test Set")
)

---
## 🏆 9. Kesimpulan

### Model Terpilih: CNN-BiLSTM

| Properti | Nilai |
|---|---|
| Arsitektur | Conv1D(128, k=3) → MaxPool → BiLSTM(64) → BiLSTM(32) → Dense(128) |
| Parameter | ~2.12 juta ✅ (< 10 juta) |
| Optimizer | Adam (lr=1e-3, ReduceLROnPlateau) |
| Loss | Binary Crossentropy |
| Imbalance | Balanced class weights |
| Regularisasi | SpatialDropout1D + Dropout(0.3) + EarlyStopping |

### Output Files
| File | Keterangan |
|---|---|
| `cnn_bilstm_tokenizer.pkl` | Tokenizer untuk inferensi |
| `cnn_bilstm_HS_best.h5` | Best model untuk label HS |
| `cnn_bilstm_Abusive_best.h5` | Best model untuk label Abusive |
| `cnn_bilstm_results.csv` | Ringkasan metrik |
| `training_curves.png` | Kurva loss & AUC |
| `confusion_matrix.png` | Confusion matrix |